In [3]:
# ============================================================
# Employee Attrition Prediction - Data Preprocessing
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [5]:
data_path = r"C:\Users\Pratyush\OneDrive\Desktop\ML-DE Final Project\WA_Fn-UseC_-HR-Employee-Attrition.csv"


df = pd.read_csv(data_path)
print("Original shape:", df.shape)
df.head()

Original shape: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


In [6]:
# These columns have constant values or are just IDs
cols_to_drop = ["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"]

df = df.drop(columns=cols_to_drop)
print("Shape after dropping useless columns:", df.shape)
print("\nRemaining columns:")
print(df.columns.tolist())

Shape after dropping useless columns: (1470, 31)

Remaining columns:
['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [7]:
# Convert Attrition to binary (Yes=1, No=0)
df["Attrition"] = df["Attrition"].map({"Yes": 1, "No": 0})

print("Target distribution:")
print(df["Attrition"].value_counts())
print("\nPercentage:")
print(df["Attrition"].value_counts(normalize=True).round(4) * 100)

Target distribution:
Attrition
0    1233
1     237
Name: count, dtype: int64

Percentage:
Attrition
0    83.88
1    16.12
Name: proportion, dtype: float64


In [8]:
# Create new meaningful features
df["YearsAtOtherCompanies"] = df["TotalWorkingYears"] - df["YearsAtCompany"]
df["IncomePerYearExp"] = df["MonthlyIncome"] / (df["TotalWorkingYears"] + 1)
df["RoleStability"] = df["YearsInCurrentRole"] / (df["YearsAtCompany"] + 1)
df["PromotionGap"] = df["YearsAtCompany"] - df["YearsSinceLastPromotion"]
df["AgeWhenStarted"] = df["Age"] - df["TotalWorkingYears"]

print("New features created:")
print(df[["YearsAtOtherCompanies", "IncomePerYearExp", "RoleStability", 
          "PromotionGap", "AgeWhenStarted"]].head())

New features created:
   YearsAtOtherCompanies  IncomePerYearExp  RoleStability  PromotionGap  \
0                      2        665.888889       0.571429             6   
1                      0        466.363636       0.636364             9   
2                      7        261.250000       0.000000             0   
3                      0        323.222222       0.777778             5   
4                      4        495.428571       0.666667             0   

   AgeWhenStarted  
0              33  
1              39  
2              30  
3              25  
4              21  


In [9]:
X = df.drop(columns=["Attrition"])
y = df["Attrition"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (1470, 35)
Target shape: (1470,)


In [10]:
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns ({}):".format(len(numerical_cols)))
print(numerical_cols)
print("\nCategorical columns ({}):".format(len(categorical_cols)))
print(categorical_cols)

Numerical columns (28):
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'YearsAtOtherCompanies', 'IncomePerYearExp', 'RoleStability', 'PromotionGap', 'AgeWhenStarted']

Categorical columns (7):
['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training set:", X_train.shape, y_train.shape)
print("Test set:    ", X_test.shape, y_test.shape)

print("\nTarget distribution in Train:")
print(y_train.value_counts(normalize=True).round(4) * 100)
print("\nTarget distribution in Test:")
print(y_test.value_counts(normalize=True).round(4) * 100)

Training set: (1176, 35) (1176,)
Test set:     (294, 35) (294,)

Target distribution in Train:
Attrition
0    83.84
1    16.16
Name: proportion, dtype: float64

Target distribution in Test:
Attrition
0    84.01
1    15.99
Name: proportion, dtype: float64


In [12]:
# Numerical pipeline: Scaling
# Categorical pipeline: One-Hot Encoding

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [13]:
# Fit on training data only
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed Train shape:", X_train_processed.shape)
print("Processed Test shape: ", X_test_processed.shape)

Processed Train shape: (1176, 56)
Processed Test shape:  (294, 56)


In [14]:
# Get feature names after one-hot encoding
cat_feature_names = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols)
all_feature_names = numerical_cols + list(cat_feature_names)

print("Total features after encoding:", len(all_feature_names))
print("\nSample feature names:")
print(all_feature_names[:15])

Total features after encoding: 56

Sample feature names:
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction']


In [15]:
print("Before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)

print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())
print("\nNew training shape:", X_train_smote.shape)

Before SMOTE:
Attrition
0    986
1    190
Name: count, dtype: int64

After SMOTE:
Attrition
0    986
1    986
Name: count, dtype: int64

New training shape: (1972, 56)


In [16]:
# Create processed folder if not exists
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Save processed arrays
np.save("../data/processed/X_train.npy", X_train_smote)
np.save("../data/processed/X_test.npy", X_test_processed)
np.save("../data/processed/y_train.npy", y_train_smote)
np.save("../data/processed/y_test.npy", y_test.to_numpy())

# Save feature names
joblib.dump(all_feature_names, "../data/processed/feature_names.pkl")

# Save the preprocessor (very important for future predictions)
joblib.dump(preprocessor, "../models/preprocessor.pkl")

print("All processed files saved successfully!")

All processed files saved successfully!


In [17]:
print("="*60)
print("PREPROCESSING COMPLETED")
print("="*60)

print(f"""
Summary:
---------
• Original shape          : {df.shape}
• Features after encoding : {X_train_processed.shape[1]}
• Training samples (SMOTE): {X_train_smote.shape[0]}
• Test samples            : {X_test_processed.shape[0]}
• Class balance after SMOTE: Equal (50-50)

Files saved:
• ../data/processed/X_train.npy
• ../data/processed/X_test.npy
• ../data/processed/y_train.npy
• ../data/processed/y_test.npy
• ../data/processed/feature_names.pkl
• ../models/preprocessor.pkl
""")

PREPROCESSING COMPLETED

Summary:
---------
• Original shape          : (1470, 36)
• Features after encoding : 56
• Training samples (SMOTE): 1972
• Test samples            : 294
• Class balance after SMOTE: Equal (50-50)

Files saved:
• ../data/processed/X_train.npy
• ../data/processed/X_test.npy
• ../data/processed/y_train.npy
• ../data/processed/y_test.npy
• ../data/processed/feature_names.pkl
• ../models/preprocessor.pkl

